# **01. Download georeferenced ice charts (shapefiles)**

## Georeferenced ice charts available at the Polar View
- You can download georeferenced ice charts available at the Polar View website (https://www.icelogistics.info/arctic).
- In Polar View, the following ice centers/agencies have provided georeferenced format of Arctic ice charts:
    * Danish Meteorological Insitute (DMI)
    * US National Ice Chart (NIC) - This data will be downloaded from NSIDC/NOAA HTML link, not the polar view.
    * Canadian Ice Service (CIS)
    * Norwegian Meteorological Institute (METNO)
    * US National Oceanic and Atmospheric Administration (NOAA)

<table style="margin-left: 0; margin-right: auto;">
  <tr>
    <th>Ice Center</th>
    <th>Product ID in Polar View</th>
    <th>Regions</th>
    <th>Time span (shapfiles)</th>
    <th>Variables</th>
  </tr>
  <tr>
    <td rowspan="8">Danish Meteorological Insitute (DMI)</td>
    <td>1301</td>
    <td>Greenland Cape Farewell</td>
    <td rowspan="8">January 2024 -</td>
    <td rowspan="8">CT, CA, CB, CC, SA, SB, SC, FA, FB, FC, CN, CD</td>
  </tr>
  <tr>
    <td>1302</td>
    <td>Greenland East Central</td>
  </tr>
  <tr>
    <td>1303</td>
    <td>Greenland East South</td>
  </tr>
  <tr>
    <td>1304</td>
    <td>Greenland East North</td>
  </tr>
  <tr>
    <td>1305</td>
    <td>Greenland West South</td>
  </tr>
  <tr>
    <td>1306</td>
    <td>Greenland West Central</td>
  </tr>
  <tr>
    <td>1308</td>
    <td>Greenland West North</td>
  </tr>
  <tr>
    <td>1309</td>
    <td>Greenland North</td>
  </tr>
  <tr>
    <td rowspan="5">Canadian Ice Service (CIS)</td>
    <td>2208</td>
    <td>Eastern Arctic</td>
    <td rowspan="5">January 2024 -</td>
    <td rowspan="5">CT, CA, CB, CC, SA, SB, SC, FA, FB, FC, CN, CD</td>
  </tr>
  <tr>
    <td>2215</td>
    <td>Canada East Coast</td>
  </tr>
  <tr>
    <td>2224</td>
    <td>Great Lakes</td>
  </tr>
  <tr>
    <td>2227</td>
    <td>Hudson Bay</td>
  </tr> 
  <tr>
    <td>2230</td>
    <td>Western Arctic</td>
  </tr> 
  <tr>
    <td rowspan="1">Norwegian Meteorological Institute (METNO)</td>
    <td>1400</td>
    <td>European Arctic</td>
    <td rowspan="1">August 2024 -</td>
    <td rowspan="1">CT, CA, CB, CC, SA, SB, SC, FA, FB, FC, CN, CD</td>
  </tr>
  <tr>
    <td rowspan="1">National Oceanic and Atmospheric Administration (NOAA)</td>
    <td>1900</td>
    <td>Alaska</td>
    <td rowspan="1">August 2024 -</td>
    <td rowspan="1">CT, CA, CB, CC, SA, SB, SC, FA, FB, FC, CN, CD</td>
  </tr>
  <tr>
    <td rowspan="1">US National Ice Center (NIC)</td>
    <td><a href="https://noaadata.apps.nsidc.org/NOAA/G10013/north/">NSIDC/NOAA</a></td>
    <td>Alaska</td>
    <td rowspan="1">2003 -</td>
    <td rowspan="1">CT, CA, CB, CC, SA, SB, SC, FA, FB, FC, CN, CD</td>
  </tr>
    
</table>



## Necessary functions

In [2]:
import gportal
import json
import os, glob
import getpass
import requests
from datetime import datetime, timedelta
import zipfile
import sys

import requests
from tqdm import tqdm
from bs4 import BeautifulSoup

# Folder to download files
scratch = "C:\\Users\\yoko2261\\OneDrive - UCB-O365\\CAIG\\Ice_chart\\"

In [3]:
def get_region(product_id):
    # Get the region name corresponding to product ids
    if product_id == 1301:
        region = "CapeFarewell"
    elif product_id == 1302:
        region = "CentralEast"
    elif product_id == 1303:
        region = "SouthEast"
    elif product_id == 1304:
        region = "NorthEast"
    elif product_id == 1305:
        region = "SouthWest"
    elif product_id == 1306:
        region = "CentralWest"
    elif product_id == 1308:
        region = "NorthWest"
    elif product_id == 1309:
        region = "North"
    elif product_id == 1900:
        region = "AlaskaCT"
    elif product_id == 1400:
        region = "EuropeanArctic"
    elif product_id == 2208:
        region = "EasternArctic"
    elif product_id == 2215:
        region = "CanadaEastCoast"
    elif product_id == 2224:
        region = "GreatLakes"
    elif product_id == 2227:
        region = "HudsonBay"
    elif product_id == 2230:
        region = "WesternArctic"
    elif product_id == 10013:
        region = "north"
    else:
        raise ValueError(f"Product id {product_id} is not correct!")
    return region

def get_agency(product_id):
    # Get the agency (ice center) name corresponding to product ids
    if str(product_id)[:2] == "13":
        agency = "DMI"
    elif str(product_id)[:2] == "22":
        agency = "CIS"
    elif str(product_id) == "1400":
        agency = "METNO"
    elif str(product_id) == "1900":
        agency = "NOAA"
    elif str(product_id) == "10013":
        agency = "NIC"
    else:
        raise ValueError(f"Product id {product_id} is not correct!")
    return agency        

def get_DMI_link(product_id, date0, ddays):
    # Get the link to download DMI ice chart products
    # - product_id: product id in polar view
    # - date0: start date for date filtering
    # - ddays: how many days you want to apply filtering after date0 in days (e.g., 7 = 7 days)
    
    region = get_region(product_id)
    date1 = (datetime.strptime(date0, "%Y%m%d") + timedelta(days = 0)).strftime("%Y-%m-%d")
    date2 = (datetime.strptime(date0, "%Y%m%d") + timedelta(days = ddays)).strftime("%Y-%m-%d")

    page_url = f"https://www.icelogistics.info/ice-charts?iceChartProductId={product_id}&dateRange={date1}%2F{date2}"

    response = requests.get(page_url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 1. Find the table (first one, or use class/id if known)
    table = soup.find("table")
    
    # 2. Get all rows
    try:
        rows = table.find_all("tr")
    except:
        rows = []
    
    # 3. Loop through rows and extract each cell
    times = []    
    for row in rows:
        cells = row.find_all("td")
        cell_text = [c.get_text(strip=True) for c in cells]
        if len(cell_text) > 0:
            if cell_text[-1][:8] == "Download": # == "DownloadSelect a format":
                times.append(cell_text[0][-12:])
            # elif cell_text[-1] == "Download":
            #     times.append(cell_text[0][-12:])

    # 4. Get links of all filenames
    links = []
    dates = []
    for t0 in times:    
        data_url = f"https://www.polarview.aq/images/ilp/products/13_DMI/{product_id}_{region}/{t0[:8]}/{t0}_{region}_RIC.shp.zip"
        response = requests.get(data_url)
        if response.status_code == 200:
            links.append(data_url)
            dates.append(t0[:8])

    return links, dates

def get_chart_link(product_id, date0, ddays):
    # Get the link to download other ice chart products
    # - product_id: product id in polar view (NIC ice chart: product id 10013)
    # - date0: start date for date filtering
    # - ddays: how many days you want to apply filtering after date0 in days (e.g., 7 = 7 days)
    
    region = get_region(product_id)

    links = []
    dates = []
    for d in range(0, ddays):
        date1 = (datetime.strptime(date0, "%Y%m%d") + timedelta(days = d)).strftime("%Y%m%d")
        ## NOAA
        if product_id == 1900:
            data_url = f"https://www.polarview.aq/images/ilp/products/19_NOAA/{product_id}_{region}/{date1}/full_latest.zip"
        ## METNO
        elif product_id == 1400:
            data_url = f"https://www.polarview.aq/images/ilp/products/14_METNO/{product_id}_{region}/{date1}/NIS_arctic_latest_pl_a.zip"
        ## CIS
        elif product_id == 2208:
            data_url = f"https://www.polarview.aq/images/ilp/products/22_CIS/{product_id}_{region}_SIGRID3/{date1}/rgc_a11_{date1}_CEXPREA.zip"
        elif product_id == 2215:
            data_url = f"https://www.polarview.aq/images/ilp/products/22_CIS/{product_id}_{region}_SIGRID3/{date1}/rgc_a12_{date1}_CEXPREC.zip"
        elif product_id == 2224:
            data_url = f"https://www.polarview.aq/images/ilp/products/22_CIS/{product_id}_{region}_SIGRID3/{date1}/rgc_a13_{date1}_CEXPRGL.zip"
        elif product_id == 2227:
            data_url = f"https://www.polarview.aq/images/ilp/products/22_CIS/{product_id}_{region}_SIGRID3/{date1}/rgc_a09_{date1}_CEXPRHB.zip"
        elif product_id == 2230:
            data_url = f"https://www.polarview.aq/images/ilp/products/22_CIS/{product_id}_{region}_SIGRID3/{date1}/rgc_a10_{date1}_CEXPRWA.zip"
        ## NIC
        elif product_id == 10013:
            data_url = f"https://noaadata.apps.nsidc.org/NOAA/G{product_id}/{region}/{date1[:4]}/ARCTIC{date1}.zip"

        response = requests.get(data_url)
        if response.status_code == 200:
            dates.append(date1)
            links.append(data_url)

    return links, dates


In [7]:
# Set up start date & time interval (look for ice charts for every ddays)
date0 = "20250222"
ddays = 7

# What product ID you want to download?
product_list = [1301, 1302, 1303, 1304, 1305, 1306, 1308, 1309,
                2208, 2215, 2224, 2227, 2230,
                1400,
                1900,
                10013
               ]

while date0[:6] != "202601": # Set the ending date for searching
    print(date0)
    for product_id in product_list:
        # Get the agency name for this product and make a folder to save ice charts in shapefiles
        agency = get_agency(product_id)        
        path = os.path.join(scratch, agency)
        if not os.path.exists(path):
            os.makedirs(path)

        # Get the available links to download files
        if str(product_id)[:2] == "13": # If DMI product - use "get_DMI_link" function
            links, dates = get_DMI_link(product_id, date0, ddays)
        else:
            links, dates = get_chart_link(product_id, date0, ddays)

        # Download zipped shapefiles from the links and unzip those files
        for n in range(0, len(links)):
            data_link = links[n]
            date_target = dates[n]
            out_file = os.path.join(path, f"{agency}_{product_id}_{date_target}.zip")

            if not os.path.exists(out_file):            
                response = requests.get(data_link)
                if response.status_code == 200:
                    # Download zipped files
                    with open(out_file, 'wb') as file:
                        file.write(response.content)

                    # Unzip files
                    with zipfile.ZipFile(out_file, 'r') as zip_ref:
                        zip_ref.extractall(os.path.join(path, out_file.replace(".zip", "")))
                        
                    os.remove(out_file)
                    
        if len(links) > 0:
            print(f'## Files downloaded successfully {product_id}')

    # Next search date (date0) becomes the day after ddays
    date0 = (datetime.strptime(date0, "%Y%m%d") + timedelta(days = ddays)).strftime("%Y%m%d")

20250222
## Files downloaded successfully 1302
## Files downloaded successfully 1303
## Files downloaded successfully 1304
## Files downloaded successfully 1305
## Files downloaded successfully 1306
## Files downloaded successfully 1308
## Files downloaded successfully 1309
## Files downloaded successfully 2208
## Files downloaded successfully 2215
## Files downloaded successfully 2224
## Files downloaded successfully 2227
## Files downloaded successfully 2230
## Files downloaded successfully 1400
## Files downloaded successfully 1900
## Files downloaded successfully 10013
20250301
## Files downloaded successfully 1302
## Files downloaded successfully 1303
## Files downloaded successfully 1304
## Files downloaded successfully 1305
## Files downloaded successfully 1306
## Files downloaded successfully 1308
## Files downloaded successfully 1309
## Files downloaded successfully 2208
## Files downloaded successfully 2215
## Files downloaded successfully 2224
## Files downloaded successfull